In [1]:
import pandas as pd
import sqlite3

In [2]:
# 1. Connect and temporarily turn OFF strict rules to allow messy raw data to load
conn = sqlite3.connect('../olist.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = OFF;") 

# 2. Define the strict architecture
schema = """
CREATE TABLE category_translation (product_category_name TEXT PRIMARY KEY, product_category_name_english TEXT);
CREATE TABLE products (product_id TEXT PRIMARY KEY, product_category_name TEXT, product_name_lenght INTEGER, product_description_lenght INTEGER, product_photos_qty INTEGER, product_weight_g REAL, product_length_cm REAL, product_height_cm REAL, product_width_cm REAL, FOREIGN KEY (product_category_name) REFERENCES category_translation(product_category_name));
CREATE TABLE sellers (seller_id TEXT PRIMARY KEY, seller_zip_code_prefix TEXT, seller_city TEXT, seller_state TEXT);
CREATE TABLE customers (customer_id TEXT PRIMARY KEY, customer_unique_id TEXT, customer_zip_code_prefix TEXT, customer_city TEXT, customer_state TEXT);
CREATE TABLE geolocation (geolocation_zip_code_prefix TEXT, geolocation_lat REAL, geolocation_lng REAL, geolocation_city TEXT, geolocation_state TEXT);
CREATE TABLE orders (order_id TEXT PRIMARY KEY, customer_id TEXT, order_status TEXT, order_purchase_timestamp TEXT, order_approved_at TEXT, order_delivered_carrier_date TEXT, order_delivered_customer_date TEXT, order_estimated_delivery_date TEXT, FOREIGN KEY (customer_id) REFERENCES customers(customer_id));
CREATE TABLE order_items (order_id TEXT, order_item_id INTEGER, product_id TEXT, seller_id TEXT, shipping_limit_date TEXT, price REAL, freight_value REAL, PRIMARY KEY (order_id, order_item_id), FOREIGN KEY (order_id) REFERENCES orders(order_id), FOREIGN KEY (product_id) REFERENCES products(product_id), FOREIGN KEY (seller_id) REFERENCES sellers(seller_id));
CREATE TABLE payments (order_id TEXT, payment_sequential INTEGER, payment_type TEXT, payment_installments INTEGER, payment_value REAL, PRIMARY KEY (order_id, payment_sequential), FOREIGN KEY (order_id) REFERENCES orders(order_id));
CREATE TABLE reviews (review_id TEXT, order_id TEXT, review_score INTEGER, review_comment_title TEXT, review_comment_message TEXT, review_creation_date TEXT, review_answer_timestamp TEXT, PRIMARY KEY (review_id, order_id), FOREIGN KEY (order_id) REFERENCES orders(order_id));
"""
cursor.executescript(schema)

# 3. Load data respecting the schema
tables = {
    'category_translation': '../data/product_category_name_translation.csv',
    'products': '../data/olist_products_dataset.csv',
    'sellers': '../data/olist_sellers_dataset.csv',
    'customers': '../data/olist_customers_dataset.csv',
    'geolocation': '../data/olist_geolocation_dataset.csv',
    'orders': '../data/olist_orders_dataset.csv',
    'order_items': '../data/olist_order_items_dataset.csv',
    'payments': '../data/olist_order_payments_dataset.csv',
    'reviews': '../data/olist_order_reviews_dataset.csv'
}

for table_name, filepath in tables.items():
    df = pd.read_csv(filepath)
    df.to_sql(table_name, conn, if_exists='append', index=False)
    print(f"Loaded {table_name}: {len(df)} rows")

conn.commit()
conn.close()

Loaded category_translation: 71 rows
Loaded products: 32951 rows
Loaded sellers: 3095 rows
Loaded customers: 99441 rows
Loaded geolocation: 1000163 rows
Loaded orders: 99441 rows
Loaded order_items: 112650 rows
Loaded payments: 103886 rows
Loaded reviews: 99224 rows


In [4]:
conn = sqlite3.connect('../olist.db')

print("1. ROW COUNTS")
tables = [
    'category_translation', 'products', 'sellers', 'customers', 
    'geolocation', 'orders', 'order_items', 'payments', 'reviews'
]

for table in tables:
    count_df = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table}", conn)
    print(f"{table}: {count_df['count'][0]} rows")


print("\n2. SCHEMA VERIFICATION (Customers Table)")

schema_df = pd.read_sql_query("PRAGMA table_info(customers)", conn)
print(schema_df[['name', 'type']])

print("\n3. ZIP CODE INTEGRITY CHECK")

zip_df = pd.read_sql_query("SELECT customer_zip_code_prefix FROM customers LIMIT 5", conn)
print(zip_df)
print(f"Pandas sees this column as: {zip_df['customer_zip_code_prefix'].dtype}")

conn.close()

1. ROW COUNTS
category_translation: 71 rows
products: 32951 rows
sellers: 3095 rows
customers: 99441 rows
geolocation: 1000163 rows
orders: 99441 rows
order_items: 112650 rows
payments: 103886 rows
reviews: 99224 rows

2. SCHEMA VERIFICATION (Customers Table)
                       name  type
0               customer_id  TEXT
1        customer_unique_id  TEXT
2  customer_zip_code_prefix  TEXT
3             customer_city  TEXT
4            customer_state  TEXT

3. ZIP CODE INTEGRITY CHECK
  customer_zip_code_prefix
0                    14409
1                     9790
2                     1151
3                     8775
4                    13056
Pandas sees this column as: str
